# Data Quality Note

- Keep one review per order, choosing the latest by answer timestamp and then creation date.
- Keep only delivered orders.
- Use unique customer ID, not customer ID, to identify customers.
- Keep only orders from the steady period, starting January 2017.

**Cleaning Summary**
Started with 99,441 orders and 95,560 remain after cleaning (96.1%)

In [2]:
%load_ext sql
%sql duckdb:///../data/olist.duckdb
%config SqlMagic.displaylimit = 20

RuntimeError: (_duckdb.IOException) IO Error: Could not set lock on file "/Users/mac/Projects/olist-customer-experience/data/olist.duckdb": Conflicting lock is held in /opt/anaconda3/bin/python3.12 (PID 82281) by user mac. See also https://duckdb.org/docs/current/connect/concurrency
(Background on this error at: https://sqlalche.me/e/21/e3q8)


# Data Quality Checks

This notebook checks the raw Olist tables before cleaning. 

In [ ]:
%%sql

SELECT
    (
        SELECT COUNT(*)
        FROM (
            SELECT order_id
            FROM order_reviews
            GROUP BY order_id
            HAVING COUNT(*) > 1
        ) AS dup_orders
    ) AS orders_with_duplicates_review,
    (
        SELECT COUNT(*)
        FROM (
            SELECT review_id
            FROM order_reviews
            GROUP BY review_id
            HAVING COUNT(*) > 1
        ) AS dup_reviews
    ) AS review_on_multiple_orders

**Duplicate reviews**

Neither order_id nor review_id is unique in the reviews table.

Decision: the reviews must be reduced to one per order before joining them to other tables, otherwise orders will be counted more than once.

In [ ]:
%%sql

SELECT
    COUNT(*) AS orders_with_mult_reviews,
    COUNT(*) FILTER (WHERE min_score <> max_score) AS scores_differ
FROM (
    SELECT
        order_id,
        MIN(review_score) AS min_score,
        MAX(review_score) AS max_score
    FROM order_reviews
    GROUP BY order_id
    HAVING COUNT(*) > 1
) AS multi_review_orders

**Do the duplicate reviews have different scores?**

Decision: which review is kept changes the score, so a clear rule is needed for choosing one.

In [ ]:
%%sql

SELECT order_id, review_id, review_score, review_answer_timestamp
FROM order_reviews
WHERE order_id IN (
    SELECT order_id FROM order_reviews
    GROUP BY order_id HAVING COUNT(*) > 1
)
ORDER BY order_id, review_answer_timestamp DESC
LIMIT 10

**Sample of orders with several reviews**

In [ ]:
%%sql

SELECT
    COUNT(*) AS rows_after,
    COUNT(DISTINCT order_id) AS distinct_orders
FROM (
    SELECT *
    FROM order_reviews
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY order_id
        ORDER BY review_answer_timestamp DESC, review_creation_date DESC
    ) = 1
) 


Sorting by answer timestamp first and creation date second to break ties.

In [ ]:
%%sql

SELECT 'orders and customers' AS join_check, COUNT(*) AS orphans
FROM orders AS o
LEFT JOIN customers AS c ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL

UNION ALL

SELECT 'order_items and orders', COUNT(*)
FROM order_items AS i
LEFT JOIN orders AS o ON i.order_id = o.order_id
WHERE o.order_id IS NULL

UNION ALL

SELECT 'order_items and products', COUNT(*)
FROM order_items AS i
LEFT JOIN products AS p ON i.product_id = p.product_id
WHERE p.product_id IS NULL

UNION ALL

SELECT 'order_items and sellers', COUNT(*)
FROM order_items AS i
LEFT JOIN sellers AS s ON i.seller_id = s.seller_id
WHERE s.seller_id IS NULL

UNION ALL

SELECT 'order_reviews and orders', COUNT(*)
FROM order_reviews AS r
LEFT JOIN orders AS o ON r.order_id = o.order_id
WHERE o.order_id IS NULL

UNION ALL

SELECT 'orders with no review', COUNT(*)
FROM orders AS o
LEFT JOIN order_reviews AS r ON o.order_id = r.order_id
WHERE r.order_id IS NULL

UNION ALL

SELECT 'orders with no items', COUNT(*)
FROM orders AS o
LEFT JOIN order_items AS i ON o.order_id = i.order_id
WHERE i.order_id IS NULL


**Do the tables link up correctly?**

Every review belongs to a real order. However, 768 orders have no review and 775 orders have no items.

Decision: the keys are reliable and joins are safe. Orders without a review or without items will drop.

In [ ]:
%%sql

select count(distinct customer_id) as customer_ids,
count (distinct customer_unique_id) as customer_unique_ids
from customers

customer ID does not identify a person

In [ ]:
%%sql
select max(n)
from (
    select customer_id, count(*) as n
    from orders
    group by customer_id
)

customer ID works as an order level key, not as a customer.

In [ ]:
%%sql
select count(*)
from (
    select c.customer_unique_id
    from orders as o
    join customers as c using (customer_id)
    group by c.customer_unique_id
    having count(*) > 1
)

**Real customers with more than one order**

Finding: 2,997 real customers, identified by unique customer ID, placed two or more orders.

In [ ]:
%%sql

SELECT
    (
        SELECT MAX(n)
        FROM (
            SELECT customer_id, COUNT(*) AS n
            FROM orders
            GROUP BY customer_id
        ) AS order_per_customer_id
    ) AS max_orders_per_customer_id,
    (
        SELECT COUNT(*)
        FROM (
            SELECT c.customer_unique_id
            FROM orders AS o
            JOIN customers AS c USING (customer_id)
            GROUP BY c.customer_unique_id
            HAVING COUNT(*) > 1
        ) AS repeat_customers
    ) AS people_with_repeat_orders

In [ ]:
%%sql

SELECT
    order_status,
    COUNT(*) AS orders,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM orders
GROUP BY order_status
ORDER BY orders DESC

**Order status breakdown**

In [ ]:
%%sql

SELECT
    DATE_TRUNC('month', order_purchase_timestamp) AS month,
    COUNT(*) AS orders
FROM orders
GROUP BY month
ORDER BY month

start the analysis from January 2017. 